# Documentação da Base de Dados Sintética — Rede de Conexões SWITAB

## 1. Visão Geral
Este documento descreve a estrutura, o dicionário de dados e a modelagem topológica do arquivo `base_conexoes_switab_ficticia.csv`. A base foi projetada para simular as respostas do formulário de mapeamento de redes sociais acadêmicas do programa **SWITAB**, permitindo a execução direta de algoritmos de **Detecção de Comunidades** (como *Louvain*, *Leiden*, *Label Propagation* e *Girvan-Newman*) e análises de centralidade.

* **Total de Registros (Nós $V$):** 181 participantes.
* **Distribuição Institucional (Subgrafos):**
  * **UNICAP:** 71 participantes
  * **UFPE:** 37 participantes
  * **UFRPE:** 30 participantes
  * **UPE:** 22 participantes
  * **Cesar School:** 21 participantes

---

## 2. Dicionário de Dados

A tabela abaixo detalha cada coluna presente no arquivo CSV, seu tipo de dado e sua função na modelagem do grafo $G = (V, E)$:

| Coluna | Tipo | Descrição / Valores Possíveis | Papel na Rede |
| :--- | :--- | :--- | :--- |
| `id_aluna` | `string` | Identificador único sequencial (`ALUNA_001` a `ALUNA_181`). | **Chave Primária do Nó ($v \in V$)** |
| `nome_completo` | `string` | Nome completo da participante conforme lista institucional. | Rótulo de exibição do nó |
| `instituicao` | `string` | Instituição de ensino (`UPE`, `UFPE`, `UFRPE`, `UNICAP`, `Cesar School`). | Partição de componente conexo |
| `semestre_atual` | `string` | Faixa de período (`1º ao 2º`, `3º ao 4º`, `5º ao 6º`, `7º período ou superior`). | Atributo de homofilia / senioridade |
| `turno` | `string` | Turno de estudo (`Matutino`, `Vespertino`, `Noturno`, `Integral`). | Atributo de homofilia espacial |
| `modalidade_ingresso` | `string` | Forma de ingresso (`Ampla Concorrência`, `Ações Afirmativas / Cotas`, `Transferência / Segunda Graduação`, `Vestibular seriado (SSA)`). | Metadado sociodemográfico |
| `regime_dedicacao` | `string` | Dedicação acadêmica (`Exclusivamente estudante`, `Estudante e estagiária/trabalhadora`, `Estudante e bolsista`). | Metadado de disponibilidade |
| `participacao_projetos_p8` | `string` | Vínculo principal em grupos (`PIBIC/PIBITI`, `Extensão/Meninas em STEM`, `Empresa Júnior/CA/Monitoria`, `Apenas disciplinas`). | Atributo estrutural de agrupamento |
| `permanencia_campus` | `string` | Tempo de permanência física além das aulas. | Indicador de proximidade física |
| `colegas_conectadas_ids` | `string` | Lista de IDs das colegas conectadas, separados por `; ` (ex: `ALUNA_004; ALUNA_012`). | **Lista de Adjacência (Arestas $E$)** |
| `colegas_conectadas_nomes` | `string` | Lista nominal correspondente aos IDs conectados, separados por `; `. | Referência legível das arestas |
| `projetos_em_conjunto` | `string` | Projetos ou entidades compartilhadas com as colegas conectadas (separados por `; `). | Contexto da aresta |
| `fonte_duvidas_primeiro` | `string` | Canal prioritário para resolução de dúvidas acadêmicas ou burocráticas. | Indicador de fluxo de informação |
| `frequencia_interacao` | `string` | Frequência de contato (`Rara`, `Ocasional`, `Frequente`, `Muito frequente`). | **Componente de Peso da Aresta ($w_{uv}$)** |
| `foco_contato` | `string` | Natureza da interação (`Estudo para provas`, `Trabalhos práticos`, `Suporte emocional`, `Carreira/estágios`). | Tipologia do vínculo (multiplex) |
| `confianca_tecnica_1_a_5` | `int` | Escala Likert ($1$ a $5$) de confiança técnica e responsabilidade. | **Componente de Peso da Aresta ($w_{uv}$)** |
| `reciprocidade_percebida` | `string` | Percepção de apoio mútuo (`Sim, com certeza`, `Talvez / Parcialmente`, `Não`). | Indicador de simetria do vínculo |
| `pessoa_mais_conectada` | `string` | Nome indicado pela respondente como principal referência/hub do curso. | *Ground truth* para validação de hubs |
| `conforto_compartilhar_1_a_5` | `int` | Escala Likert ($1$ a $5$) de segurança psicológica/emocional. | **Componente de Peso da Aresta ($w_{uv}$)** |
| `observacoes_finais` | `string` | Relato qualitativo livre sobre integração e rede de apoio. | Entrada para análise textual (PLN) |

---

## 3. Premissas da Geração Sintética (Comportamento Topológico)

Para garantir que os algoritmos de detecção de comunidades encontrem estruturas realistas e não apenas ruído aleatório, a geração das arestas seguiu três regras topológicas:

1. **Isolamento Institucional:** Seguindo a lógica de ramificação do formulário original, as conexões ocorrem estritamente entre participantes da mesma `instituicao`, formando **5 subgrafos independentes**.
2. **Formação de Comunidades por Homofilia (70\% / 30\%):**
   * **70\% das conexões** de cada aluna foram direcionadas preferencialmente a colegas que compartilham o mesmo `turno` ou a mesma `participacao_projetos_p8`, induzindo a formação de *clusters* densos com alta modularidade ($Q$).
   * **30\% das conexões** foram distribuídas entre outros grupos da mesma instituição, atuando como **arestas de ponte (*bridges*)** essenciais para métricas de intermediação (*Betweenness Centrality*).
3. **Ponderamento dos Vínculos:** Os valores das escalas de confiança técnica, conforto emocional e frequência foram correlacionados positivamente para permitir a construção de matrizes de adjacência ponderadas.

---

## 4. Pipeline Recomendado para Execução

Para extrair as comunidades a partir deste arquivo CSV, recomenda-se o seguinte fluxo analítico:

1. **Particionamento por Instituição:** Filtrar o *DataFrame* por cada uma das 5 instituições antes de instanciar o grafo, evitando distorções nas métricas de distância e densidade causadas por componentes desconexos.
2. **Cálculo do Peso das Arestas ($w_{uv}$):** Normalizar e combinar as colunas numéricas de vínculo:
   $$w_{uv} = \frac{\text{Confiança Técnica} + \text{Conforto Compartilhar} + \text{Score Frequência}}{3}$$
3. **Execução dos Algoritmos de Comunidade:**
   * Converter o grafo direcionado (`DiGraph`) em não direcionado (`Graph`) somando ou fazendo a média dos pesos recíprocos para algoritmos baseados em otimização de modularidade (**Louvain** e **Leiden**).
   * Avaliar a **Modularidade ($Q$)** obtida e cruzar os *clusters* detectados com os metadados (`turno`, `semestre_atual` e `participacao_projetos_p8`) para interpretar os fatores que mais unem as alunas em cada universidade.

In [49]:
%pip install --upgrade nbformat

Note: you may need to restart the kernel to use updated packages.


In [38]:
%pip install networkx

Note: you may need to restart the kernel to use updated packages.


In [39]:
%pip install plotly

Note: you may need to restart the kernel to use updated packages.


In [40]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [41]:
import networkx as nx
import pandas as pd

import math
import plotly.graph_objects as go

In [42]:
df = pd.read_csv(r"data\base_conexoes_switab_ficticia.csv")
G = nx.DiGraph()

In [43]:

# Adiciona nós com atributos
for _, row in df.iterrows():
    G.add_node(
        row["id_aluna"],
        instituicao=row["instituicao"],
        turno=row["turno"],
        semestre=row["semestre_atual"],
    )



In [44]:
# Adiciona arestas ponderadas pela confiança técnica e frequência
mapa_freq = {
    "Rara (apenas em período de avaliações)": 1,
    "Ocasional (algumas vezes no mês)": 2,
    "Frequente (semanalmente)": 3,
    "Muito frequente (diariamente)": 4,
}

for _, row in df.iterrows():
    origem = row["id_aluna"]
    peso = (
        row["confianca_tecnica_1_a_5"] + mapa_freq[row["frequencia_interacao"]]
    ) / 2.0
    if pd.notna(row["colegas_conectadas_ids"]):
        for destino in row["colegas_conectadas_ids"].split("; "):
            G.add_edge(origem, destino, weight=peso)



In [45]:
# Exemplo: rodar Louvain na sub-rede da UPE
G_upe = G.subgraph(
    df[df["instituicao"] == "UPE"]["id_aluna"]
).to_undirected()
comunidades_upe = nx.community.louvain_communities(G_upe, weight="weight")
print(f"Comunidades detectadas na UPE: {len(comunidades_upe)}")

Comunidades detectadas na UPE: 3


In [46]:
print(df.columns)

Index(['id_aluna', 'nome_completo', 'instituicao', 'semestre_atual', 'turno',
       'modalidade_ingresso', 'regime_dedicacao', 'participacao_projetos_p8',
       'permanencia_campus', 'colegas_conectadas_nomes',
       'colegas_conectadas_ids', 'projetos_em_conjunto',
       'fonte_duvidas_primeiro', 'frequencia_interacao', 'foco_contato',
       'confianca_tecnica_1_a_5', 'reciprocidade_percebida',
       'pessoa_mais_conectada', 'conforto_compartilhar_1_a_5',
       'observacoes_finais'],
      dtype='str')


In [47]:
G = nx.from_pandas_edgelist(
    df,
    source="id_aluna",
    target="colegas_conectadas_ids",
    create_using=nx.DiGraph()
)

In [50]:
# Layout reproduzível: as mesmas posições a cada execução
pos = nx.spring_layout(G, seed=42)

# Arestas
edge_x, edge_y = [], []
setas = []

for origem, destino in G.edges():
    x0, y0 = pos[origem]
    x1, y1 = pos[destino]

    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

    # Indica a direção da conexão
    setas.append({
        "x": x1, "y": y1,
        "ax": x0, "ay": y0,
        "xref": "x", "yref": "y",
        "axref": "x", "ayref": "y",
        "showarrow": True,
        "arrowhead": 2,
        "arrowsize": 1,
        "arrowwidth": 1,
        "arrowcolor": "#9AA0A6"
    })

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=edge_x,
    y=edge_y,
    mode="lines",
    line=dict(width=1, color="#9AA0A6"),
    hoverinfo="skip"
))

nos = list(G.nodes())
grau_entrada = [G.in_degree(no) for no in nos]
tamanho = [10 + 3 * math.sqrt(G.degree(no)) for no in nos]

fig.add_trace(go.Scatter(
    x=[pos[no][0] for no in nos],
    y=[pos[no][1] for no in nos],
    mode="markers",
    marker=dict(
        size=tamanho,
        color=grau_entrada,
        colorscale="Viridis",
        showscale=True,
        colorbar=dict(title="Conexões<br>recebidas"),
        line=dict(width=1, color="white")
    ),
    customdata=[
        [str(no), G.in_degree(no), G.out_degree(no)]
        for no in nos
    ],
    hovertemplate=(
        "<b>ID: %{customdata[0]}</b><br>"
        "Conexões recebidas: %{customdata[1]}<br>"
        "Conexões feitas: %{customdata[2]}"
        "<extra></extra>"
    )
))

fig.update_layout(
    title="Rede de conexões entre alunas",
    annotations=setas,
    hovermode="closest",
    showlegend=False,
    template="plotly_white",
    margin=dict(l=20, r=20, t=60, b=20),
    xaxis=dict(visible=False),
    yaxis=dict(visible=False, scaleanchor="x", scaleratio=1)
)

fig.show()
fig.show(renderer="browser")

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed